In [ ]:
from pathlib import Path
import tifffile as tf
import numpy as np 
import histomicstk as htk
import matplotlib.pyplot as plt
import skimage
import cv2
import pickle 
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import MinMaxScaler
from scipy.ndimage import gaussian_filter

In [ ]:
key = "" # sample identifier

In [ ]:
registered_img_folder_path = Path(r"") # path to registered images

In [ ]:
color_deconvolution_path = r"" # path to color deconvolution results

In [ ]:
Path.mkdir(Path(color_deconvolution_path), exist_ok=True, parents=True)

In [ ]:
color_images_paths = [f for f in registered_img_folder_path.glob('*tif') if 'Cycle' not in str(f) and 'registered' in str(f) and 'interstitium' not in str(f)]
color_images_rgb_dict = {}
for path in color_images_paths:

    stain = str(path).split('\\')[-1].split('_')[0]
    print(stain)
    stain_img = tf.imread(path)
    black_pixels_mask = np.all(stain_img == [0, 0, 0], axis=2)
    stain_img[black_pixels_mask] = [np.max(stain_img), np.max(stain_img), np.max(stain_img)]
    color_images_rgb_dict[stain] = stain_img

In [ ]:
od = -np.log((color_images_rgb_dict['Trichrome'] + 1.0) / 256.0)
od_mag = np.linalg.norm(od, axis=-1)
mask_out_Trichrome = od_mag >= np.percentile(od_mag, 99.9)

In [ ]:
fig, ax = plt.subplots(1)
ax.imshow(mask_out_Trichrome)

In [ ]:
with open('/'.join(str(registered_img_folder_path).split('\\')) + '/roi_coord_dict.pickle', 'rb') as f:
    roi_coord_dict = pickle.load(f)

In [ ]:
if('PAS' in list(color_images_rgb_dict.keys())):
    fig, ax = plt.subplots(1, 2)
    fig.set_figwidth(15)
    ax[0].imshow(color_images_rgb_dict['PAS'])
    ax[1].imshow(color_images_rgb_dict['Trichrome'])
else:
    fig, ax = plt.subplots(1)
    fig.set_figwidth(15)
    ax.imshow(color_images_rgb_dict['Trichrome'])

In [ ]:
roi_coord_dict

In [ ]:
T_roi = roi_coord_dict['B']
PAS_roi  = roi_coord_dict['B']

In [ ]:
if('PAS' in list(color_images_rgb_dict.keys())):
    fig, ax = plt.subplots(1, 2)
    fig.set_figheight(15)
    fig.set_figwidth(15)
    ax[0].imshow(color_images_rgb_dict['PAS'][PAS_roi[0]:PAS_roi[1], PAS_roi[2]:PAS_roi[3]])
    ax[1].imshow(color_images_rgb_dict['Trichrome'][T_roi[0]:T_roi[1], T_roi[2]:T_roi[3]])
else:
    fig, ax = plt.subplots(1)
    fig.set_figheight(5)
    fig.set_figwidth(5)
    ax.imshow(color_images_rgb_dict['Trichrome'][T_roi[0]:T_roi[1], T_roi[2]:T_roi[3]])

In [ ]:
PAS_exists = False
if('PAS' in list(color_images_rgb_dict.keys())):
    PAS_exists = True

##### Test htk PAS deconvolution

In [ ]:
if(PAS_exists):
    PAS_test = color_images_rgb_dict['PAS'][PAS_roi[0]:PAS_roi[1], PAS_roi[2]:PAS_roi[3]]
    PAS_test = np.clip(PAS_test * 255, 0, 255).astype(np.uint8)
    w_est_PAS = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(PAS_test, 255)
    PAS_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(PAS_test, w_est_PAS, 255)

    fig, ax = plt.subplots(1, 3)
    fig.set_figheight(10)
    fig.set_figwidth(30)
    ax[0].imshow(255-PAS_deconvolved.Stains[:, :, 0])
    ax[1].imshow(255-PAS_deconvolved.Stains[:, :, 1])
    ax[2].imshow(255-PAS_deconvolved.Stains[:, :, 2])

##### Test htk Trichrome deconvolution

In [ ]:
Trichrome_test = color_images_rgb_dict['Trichrome'][T_roi[0]:T_roi[1], T_roi[2]:T_roi[3]]
Trichrome_test = np.clip(Trichrome_test * 255, 0, 255).astype(np.uint8)

In [ ]:
"""w_est_Trichrome = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(Trichrome_test, 255)
Trichrome_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(Trichrome_test, w_est_Trichrome, 255)"""
if not hasattr(np, "mat"):
    np.mat = np.asmatrix

if not hasattr(np, "Inf"):
    np.Inf = np.inf

w_est_Trichrome = htk.preprocessing.color_deconvolution.rgb_separate_stains_xu_snmf(Trichrome_test[:500, :500], 255)
Trichrome_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(Trichrome_test, w_est_Trichrome, 255)


In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(255-Trichrome_deconvolved.Stains[:, :, 0])
ax[1].imshow(255-Trichrome_deconvolved.Stains[:, :, 1])
ax[2].imshow(255-Trichrome_deconvolved.Stains[:, :, 2])

##### Compute htk color deconvolution for HE, PAS and Trichrome

In [ ]:
if(PAS_exists):
    PAS =  np.clip(color_images_rgb_dict['PAS'] * 255, 0, 255).astype(np.uint8)
Trichrome =  np.clip(color_images_rgb_dict['Trichrome'] * 255, 0, 255).astype(np.uint8)

In [ ]:
if(PAS_exists):
    w_est_PAS_wsi = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(PAS, 255)
    PAS_deconvolved_wsi = htk.preprocessing.color_deconvolution.color_deconvolution(PAS, w_est_PAS_wsi, 255)

In [ ]:
if(PAS_exists):
    fig, ax = plt.subplots(1, 3)
    fig.set_figheight(10)
    fig.set_figwidth(30)
    ax[0].imshow(255-PAS_deconvolved_wsi.Stains[:, :, 0][PAS_roi[0]:PAS_roi[1], PAS_roi[2]:PAS_roi[3]])
    ax[1].imshow(255-PAS_deconvolved_wsi.Stains[:, :, 1][PAS_roi[0]:PAS_roi[1], PAS_roi[2]:PAS_roi[3]])
    ax[2].imshow(255-PAS_deconvolved_wsi.Stains[:, :, 2][PAS_roi[0]:PAS_roi[1], PAS_roi[2]:PAS_roi[3]])

In [ ]:

if not hasattr(np, "mat"):
    np.mat = np.asmatrix

if not hasattr(np, "Inf"):
    np.Inf = np.inf

w_est_Trichrome = htk.preprocessing.color_deconvolution.rgb_separate_stains_xu_snmf(Trichrome_test[:500, :500], 255)
Trichrome_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(Trichrome, w_est_Trichrome, 255)


In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(255-Trichrome_deconvolved.Stains[:, :, 0][T_roi[0]:T_roi[1], T_roi[2]:T_roi[3]])
ax[1].imshow(255-Trichrome_deconvolved.Stains[:, :, 1][T_roi[0]:T_roi[1], T_roi[2]:T_roi[3]])
ax[2].imshow(255-Trichrome_deconvolved.Stains[:, :, 2][T_roi[0]:T_roi[1], T_roi[2]:T_roi[3]])

In [ ]:
tf.imwrite(color_deconvolution_path + '/pas.tif', PAS_deconvolved_wsi.Stains[:, :, 0])

In [ ]:
tf.imwrite(color_deconvolution_path + '/trichrome_blue_non_wsi.tif', Trichrome_deconvolved.Stains[:, :, 1])

In [ ]:
tf.imwrite(color_deconvolution_path + '/trichrome_mask_out.tif', mask_out_Trichrome)